# Detección de Procesos Espaciales con la K de Ripley
### *Ejemplo Relámpago — Introducción al Análisis de Datos y Programación para el Manejo y Conservación de Recursos Naturales*

---

Este cuaderno demuestra cómo usar la función **K de Ripley** para distinguir tres tipos de procesos de puntos:

| Proceso | ¿Qué significa? | Ejemplo ecológico |
|---------|----------------|-------------------|
| 🔴 **Agrupamiento** (atracción) | Los individuos se atraen mutuamente | Plántulas bajo el dosel de un árbol madre |
| 🟢 **Aleatoriedad** (CSR) | Cada punto es independiente de los demás | Hipótesis nula estándar |
| 🔵 **Regularidad** (repulsión) | Los individuos se evitan | Territorios, competencia por recursos |

> **¿Por qué K de Ripley y no los métodos anteriores?**  
> El correlograma de Moran's I y el semivariograma trabajan con **valores continuos** en ubicaciones muestreadas.  
> La K de Ripley trabaja con **patrones de puntos**: solo importa *dónde* están los individuos, no un valor asociado.  
> Es el método estándar en ecología de comunidades, epidemiología espacial y forestería.

## 0 · Instalación de dependencias

In [ ]:
!pip install numpy scipy matplotlib --quiet

## 1 · Importaciones

In [ ]:
import numpy as np
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

## 2 · ¿Qué es la K de Ripley?

### Definición intuitiva

$K(r)$ responde a la pregunta: **¿cuántos vecinos adicionales espero encontrar dentro de un radio $r$ alrededor de un punto típico?**

$$K(r) = \frac{A}{n^2} \sum_{i} \sum_{j \neq i} \mathbf{1}(d_{ij} \leq r) \cdot w_{ij}$$

| Símbolo | Significado |
|---------|-------------|
| $A$ | área del dominio de estudio |
| $n$ | número de puntos |
| $d_{ij}$ | distancia entre puntos $i$ y $j$ |
| $w_{ij}$ | corrección de borde (ver §3) |
| $\mathbf{1}(\cdot)$ | 1 si la condición se cumple, 0 si no |

### Bajo aleatoriedad completa (CSR)

Si los puntos son un **proceso de Poisson homogéneo** (completamente aleatorio):

$$K_{\text{CSR}}(r) = \pi r^2$$

Es decir, el número esperado de vecinos crece como el área del círculo.

### La transformación L(r) — la más usada en la práctica

Para linearizar $K$ y facilitar la interpretación, se usa:

$$L(r) = \sqrt{\frac{K(r)}{\pi}} - r$$

| Valor de L(r) | Interpretación |
|--------------|---------------|
| $L(r) > 0$ | Más vecinos de lo esperado → **agrupamiento** |
| $L(r) \approx 0$ | Tanto como lo esperado → **aleatoriedad (CSR)** |
| $L(r) < 0$ | Menos vecinos de lo esperado → **regularidad** |

## 3 · Corrección de borde (edge correction)

### El problema

Un punto cerca del borde del área de estudio tiene vecinos potenciales **fuera** del dominio observado.  
Si no corregimos esto, subestimaremos K a distancias grandes.

```
  ┌────────────────────┐
  │                    │
  │     • i            │  El círculo de radio r
  │   /     \          │  alrededor de i sale
  │  /   ✓   \         │  del área de estudio →
  ◉──────────◉─────────┤  los puntos del sector
  │           \  ✗     │  exterior no son
  │            \       │  observables
  └────────────────────┘
```

### La corrección isotrópica de Ripley

Para cada par $(i, j)$, el peso es la **inversa de la fracción del círculo** centrado en $i$ con radio $d_{ij}$ que cae dentro del dominio:

$$w_{ij} = \frac{1}{p(x_i,\, d_{ij})}$$

Si el círculo está completamente dentro del área, $p = 1$ y $w = 1$.  
Si la mitad del círculo queda fuera, $p = 0.5$ y $w = 2$ — duplicamos el peso del par observable.

In [ ]:
def fraccion_circulo_en_ventana(x, y, r, xmin, xmax, ymin, ymax, n_ang=360):
    """
    Estima la fracción del círculo de radio r centrado en (x,y)
    que cae dentro del rectángulo [xmin,xmax] x [ymin,ymax].
    Usa integración numérica sobre ángulos uniformes.
    """
    angulos = np.linspace(0, 2 * np.pi, n_ang, endpoint=False)
    px = x + r * np.cos(angulos)
    py = y + r * np.sin(angulos)
    dentro = ((px >= xmin) & (px <= xmax) &
              (py >= ymin) & (py <= ymax))
    return np.mean(dentro)


def calcular_K_L(coords, radios, xmin=0, xmax=1, ymin=0, ymax=1):
    """
    Calcula K(r) y L(r) para un patrón de puntos con
    corrección isotrópica de borde.

    Parámetros
    ----------
    coords : array (n, 2)  — coordenadas de los puntos
    radios : array          — radios r a evaluar
    xmin/xmax/ymin/ymax    — límites del área de estudio

    Retorna
    -------
    K : array   — K(r) estimada
    L : array   — L(r) = sqrt(K/π) - r
    """
    n = len(coords)
    A = (xmax - xmin) * (ymax - ymin)
    dists = cdist(coords, coords)

    K = np.zeros(len(radios))
    for idx, r in enumerate(radios):
        acum = 0.0
        for i in range(n):
            for j in range(n):
                if i == j:
                    continue
                if dists[i, j] <= r:
                    p = fraccion_circulo_en_ventana(
                        coords[i, 0], coords[i, 1], dists[i, j],
                        xmin, xmax, ymin, ymax)
                    w = 1.0 / p if p > 0 else 0.0
                    acum += w
        K[idx] = (A / n**2) * acum

    L = np.sqrt(K / np.pi) - radios
    return K, L

print("✓ Funciones de K de Ripley y corrección de borde definidas.")

## 4 · Generar los tres patrones de puntos

Creamos un patrón de cada tipo sobre una ventana unitaria $[0,1]^2$:

- **Aleatorio (CSR):** proceso de Poisson homogéneo — cada punto es independiente
- **Agrupado:** proceso de Thomas — padres aleatorios con prole dispersa en torno a ellos
- **Regular:** inhibición simple secuencial (SSI) — se rechaza cualquier punto a menos de $\delta$ de uno existente

In [ ]:
np.random.seed(42)
N_TARGET = 150   # número aproximado de puntos en cada patrón

# ── 1. Patrón aleatorio (CSR) ─────────────────────────────────────────────────
csr = np.random.uniform(0, 1, (N_TARGET, 2))

# ── 2. Patrón agrupado (proceso de Thomas) ────────────────────────────────────
# Parámetros: n_padres padres, cada uno genera prole ~ Poisson(mu_prole)
# dispersados con SD = sigma_cluster
n_padres    = 15
mu_prole    = 10     # prole esperada por padre → total ≈ 150
sigma_clust = 0.06   # dispersión de la prole en torno al padre

padres = np.random.uniform(0, 1, (n_padres, 2))
prole  = []
for p in padres:
    k = np.random.poisson(mu_prole)
    offsets = np.random.normal(0, sigma_clust, (k, 2))
    pts = p + offsets
    # mantener solo los que caen dentro del área [0,1]²
    pts = pts[(pts[:, 0] >= 0) & (pts[:, 0] <= 1) &
              (pts[:, 1] >= 0) & (pts[:, 1] <= 1)]
    prole.append(pts)
agrupado = np.vstack(prole)

# ── 3. Patrón regular (inhibición simple secuencial — SSI) ────────────────────
delta    = 0.065    # distancia mínima entre puntos
max_iter = 100_000
regular  = []
intentos = 0
while len(regular) < N_TARGET and intentos < max_iter:
    candidato = np.random.uniform(0, 1, 2)
    if len(regular) == 0:
        regular.append(candidato)
    else:
        dists_cand = cdist([candidato], regular)[0]
        if dists_cand.min() >= delta:
            regular.append(candidato)
    intentos += 1
regular = np.array(regular)

print(f"Puntos generados:")
print(f"  CSR (aleatorio):   {len(csr)}")
print(f"  Agrupado (Thomas): {len(agrupado)}")
print(f"  Regular (SSI):     {len(regular)}")

## 5 · Visualizar los tres patrones

Antes de calcular K, observa a simple vista las diferencias:  
¿puedes identificar cuál es cuál solo por el mapa de puntos?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

configs = [
    (csr,      "Aleatorio (CSR)",           "#2E7D32", "o"),
    (agrupado, "Agrupado (Thomas)",          "#C62828", "o"),
    (regular,  "Regular (SSI)",              "#1565C0", "o"),
]

for ax, (pts, titulo, color, marker) in zip(axes, configs):
    ax.scatter(pts[:, 0], pts[:, 1],
               c=color, s=15, alpha=0.7, marker=marker)
    ax.set_title(f"{titulo}\nn = {len(pts)}", fontsize=13)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.set_xlabel('X'); ax.set_ylabel('Y')
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)

plt.suptitle("Tres tipos de procesos de puntos espaciales", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('01_patrones_puntos.png', dpi=150, bbox_inches='tight')
plt.show()

## 6 · Calcular K(r) y L(r) para los tres patrones

Este paso tarda ~1–2 minutos porque evalúa todos los pares para cada radio.  
Es el núcleo del análisis — no hay atajos aquí sin perder la corrección de borde.

> **Nota pedagógica:** Para muestras grandes (> 1000 puntos) conviene usar  
> `libpysal` o `pointpats`, que implementan la misma lógica en C optimizado.  
> Aquí lo hacemos en Python puro para que el algoritmo sea completamente transparente.

In [ ]:
radios = np.linspace(0.01, 0.25, 40)

print("Calculando K de Ripley (con corrección de borde)...")
print("Esto puede tardar 1–2 minutos — paciencia 🕐\n")

K_csr,      L_csr      = calcular_K_L(csr,      radios)
print("  ✓ CSR listo")
K_agrupado, L_agrupado = calcular_K_L(agrupado, radios)
print("  ✓ Agrupado listo")
K_regular,  L_regular  = calcular_K_L(regular,  radios)
print("  ✓ Regular listo")

# Teórico bajo CSR: K(r) = πr², L(r) = 0
K_teorico = np.pi * radios**2
L_teorico = np.zeros_like(radios)  # por definición

print("\n✓ Cálculo completado.")

## 7 · Envolventes de Monte Carlo

¿Cómo sé si la L(r) observada se aparta *significativamente* de la aleatoriedad?

La respuesta es simular muchos patrones CSR con el mismo número de puntos y área,  
calcular L(r) para cada simulación, y construir una **banda de referencia**.

- Si L(r) observado queda **sobre** la banda → agrupamiento significativo  
- Si L(r) observado queda **bajo** la banda → regularidad significativa  
- Si L(r) observado queda **dentro** → no se rechaza CSR

> Con `n_sims = 99` simulaciones, la banda corresponde a un nivel de significancia  
> aproximado de α = 0.02 (prueba de dos colas).

In [ ]:
n_sims = 39   # aumenta a 99 para mayor precisión (tarda más)

def envolventes_csr(n_puntos, radios, n_sims):
    """Simula n_sims patrones CSR y retorna banda min/max de L(r)."""
    L_sims = np.zeros((n_sims, len(radios)))
    for s in range(n_sims):
        pts_sim = np.random.uniform(0, 1, (n_puntos, 2))
        _, L_sim = calcular_K_L(pts_sim, radios)
        L_sims[s] = L_sim
        if (s + 1) % 10 == 0:
            print(f"  simulación {s+1}/{n_sims}")
    return L_sims.min(axis=0), L_sims.max(axis=0)

print(f"Generando {n_sims} envolventes CSR (n ≈ {N_TARGET} puntos)...")
L_env_min, L_env_max = envolventes_csr(N_TARGET, radios, n_sims)
print("✓ Envolventes listas.")

## 8 · Graficar L(r) con las envolventes

Este es el gráfico diagnóstico estándar en análisis de patrones de puntos.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

configs_L = [
    (L_csr,      csr,      "Aleatorio (CSR)",  "#2E7D32"),
    (L_agrupado, agrupado, "Agrupado",         "#C62828"),
    (L_regular,  regular,  "Regular",          "#1565C0"),
]

for ax, (L_obs, pts, titulo, color) in zip(axes, configs_L):
    # Banda de envolventes CSR
    ax.fill_between(radios, L_env_min, L_env_max,
                    alpha=0.2, color='gray', label='Envolvente CSR')
    ax.plot(radios, L_env_min, color='gray', linewidth=0.8, linestyle='--')
    ax.plot(radios, L_env_max, color='gray', linewidth=0.8, linestyle='--')

    # Línea L(r) = 0 (CSR teórico)
    ax.axhline(0, color='black', linewidth=1, linestyle=':',
               label='L(r) = 0 (CSR teórico)')

    # L(r) observado
    ax.plot(radios, L_obs, color=color, linewidth=2.5,
            label=f'L(r) observado')

    ax.set_title(f"{titulo}\nn = {len(pts)}", fontsize=13)
    ax.set_xlabel('Radio r', fontsize=12)
    if ax == axes[0]:
        ax.set_ylabel('L(r)', fontsize=12)
    ax.legend(fontsize=9, loc='upper left')
    ax.grid(True, alpha=0.2)
    ax.set_xlim(0, 0.25)

plt.suptitle("Función L de Ripley — L(r) > 0: agrupamiento  |  L(r) < 0: regularidad",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('02_L_ripley_tres_patrones.png', dpi=150, bbox_inches='tight')
plt.show()

## 9 · Panel resumen: mapa + L(r) lado a lado

Una figura de publicación que combina el mapa del patrón y su curva L(r).

In [ ]:
fig = plt.figure(figsize=(16, 9))
gs  = GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

configs_all = [
    (csr,      L_csr,      "Aleatorio (CSR)",  "#2E7D32",
     "L(r) ≈ 0 dentro de la\nenvolvente → no se rechaza CSR"),
    (agrupado, L_agrupado, "Agrupado (Thomas)", "#C62828",
     "L(r) >> 0 a escalas cortas →\nagrupamiento significativo"),
    (regular,  L_regular,  "Regular (SSI)",     "#1565C0",
     "L(r) << 0 → regularidad\n(puntos se evitan mutuamente)"),
]

for col, (pts, L_obs, titulo, color, interpretacion) in enumerate(configs_all):
    # Fila 0: mapa de puntos
    ax_mapa = fig.add_subplot(gs[0, col])
    ax_mapa.scatter(pts[:, 0], pts[:, 1],
                    c=color, s=12, alpha=0.7)
    ax_mapa.set_title(titulo, fontsize=12, fontweight='bold', color=color)
    ax_mapa.set_xlim(0, 1); ax_mapa.set_ylim(0, 1)
    ax_mapa.set_aspect('equal')
    ax_mapa.set_xticks([]); ax_mapa.set_yticks([])
    for sp in ax_mapa.spines.values():
        sp.set_color(color); sp.set_linewidth(2)

    # Fila 1: L(r)
    ax_L = fig.add_subplot(gs[1, col])
    ax_L.fill_between(radios, L_env_min, L_env_max,
                      alpha=0.2, color='gray')
    ax_L.plot(radios, L_env_min, 'gray', linewidth=0.8, linestyle='--')
    ax_L.plot(radios, L_env_max, 'gray', linewidth=0.8, linestyle='--')
    ax_L.axhline(0, color='black', linewidth=1, linestyle=':')
    ax_L.plot(radios, L_obs, color=color, linewidth=2.5)
    ax_L.set_xlabel('Radio r', fontsize=11)
    if col == 0:
        ax_L.set_ylabel('L(r)', fontsize=11)
    ax_L.set_xlim(0, 0.25)
    ax_L.grid(True, alpha=0.2)
    ax_L.set_title(interpretacion, fontsize=9.5, style='italic', pad=4)

plt.suptitle("K de Ripley — Panel Diagnóstico Completo", fontsize=14, y=1.01)
plt.savefig('03_panel_ripley_completo.png', dpi=150, bbox_inches='tight')
plt.show()

## 10 · Resultados numéricos

In [ ]:
print("=" * 62)
print("RESULTADOS — L(r) máximo y mínimo por patrón")
print("=" * 62)
print(f"  {'Patrón':<22} {'L_max':>10} {'L_min':>10}  {'Diagnóstico'}")
print(f"  {'-'*58}")
for nombre, L_obs in [("Aleatorio (CSR)", L_csr),
                       ("Agrupado",        L_agrupado),
                       ("Regular",         L_regular)]:
    lmax = np.nanmax(L_obs)
    lmin = np.nanmin(L_obs)
    if lmax > np.nanmax(L_env_max):
        diag = "Agrupamiento significativo"
    elif lmin < np.nanmin(L_env_min):
        diag = "Regularidad significativa"
    else:
        diag = "No se rechaza CSR"
    print(f"  {nombre:<22} {lmax:>10.4f} {lmin:>10.4f}  {diag}")

## 11 · Guía de interpretación

### Lo que revela cada patrón

**Patrón aleatorio (CSR)**
- L(r) oscila cerca de 0 dentro de la banda de envolventes
- No hay estructura espacial detectable → los individuos son independientes entre sí

**Patrón agrupado**
- L(r) sube claramente por encima de la banda a escalas cortas
- El pico de L(r) indica la **escala dominante del agrupamiento**
- Interpretación ecológica: dispersión limitada, facilitación, recursos concentrados

**Patrón regular**
- L(r) cae por debajo de la banda a escalas cortas
- La escala donde L(r) toca el mínimo indica la **distancia de inhibición** (equivalente a $\delta$ en SSI)
- Interpretación ecológica: competencia interespecífica, territorialidad, autotoxicidad

### Relación con los métodos anteriores del curso

| Herramienta | Dato de entrada | Pregunta que responde |
|-------------|----------------|-----------------------|
| Correlograma Moran's I | Valores continuos en puntos muestreados | ¿A qué escala hay similitud entre vecinos? |
| Semivariograma | Valores continuos en puntos muestreados | ¿Cuánto difieren los puntos según su distancia? |
| **K de Ripley** | Solo las coordenadas (patrón de puntos) | ¿Los individuos se atraen, repelen o son independientes? |

### Limitaciones importantes

- La K de Ripley asume **estacionariedad** — la intensidad del proceso es uniforme en el área  
  Si hay un gradiente ambiental (más humedad en un sector), el agrupamiento detectado  
  puede ser de **segundo orden** (heterogeneidad del hábitat), no de primer orden (interacción entre individuos)  
- El resultado depende del **tamaño del área de estudio** — siempre reportar $n$, área y rango de radios evaluados  
- Para distinguir agrupamiento de primer vs. segundo orden, se usa la **K de Ripley inhomogénea** (paso siguiente en el curso)

## 12 · Extra: experimenta con los parámetros 🔬

Cambia los parámetros del proceso de Thomas o la distancia de inhibición SSI  
y observa cómo cambia la curva L(r).

In [ ]:
# ── Experimenta aquí ─────────────────────────────────────────────────────────
N_EXP        = 120   # número de puntos
SIGMA_CLUST  = 0.04  # dispersión del proceso de Thomas — prueba 0.02, 0.10
N_PADRES_EXP = 10    # número de padres — prueba 5, 20
DELTA_REG    = 0.08  # distancia de inhibición SSI — prueba 0.04, 0.12
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(99)

# Agrupado experimental
padres_e = np.random.uniform(0, 1, (N_PADRES_EXP, 2))
prole_e  = []
for p in padres_e:
    k   = np.random.poisson(N_EXP / N_PADRES_EXP)
    pts = p + np.random.normal(0, SIGMA_CLUST, (k, 2))
    pts = pts[(pts[:,0]>=0)&(pts[:,0]<=1)&(pts[:,1]>=0)&(pts[:,1]<=1)]
    prole_e.append(pts)
agrup_e = np.vstack(prole_e) if prole_e else np.empty((0,2))

# Regular experimental
reg_e = []
for _ in range(200_000):
    if len(reg_e) >= N_EXP:
        break
    c = np.random.uniform(0, 1, 2)
    if len(reg_e) == 0 or cdist([c], reg_e)[0].min() >= DELTA_REG:
        reg_e.append(c)
reg_e = np.array(reg_e)

print(f"Agrupado experimental: {len(agrup_e)} puntos")
print(f"Regular experimental:  {len(reg_e)} puntos")
print("Calculando L(r)...")

_, L_ag_e = calcular_K_L(agrup_e, radios) if len(agrup_e) > 10 else (None, np.full_like(radios, np.nan))
_, L_rg_e = calcular_K_L(reg_e,   radios) if len(reg_e)   > 10 else (None, np.full_like(radios, np.nan))

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, (L_o, color, titulo) in zip(axes, [
        (L_ag_e, "#C62828", f"Agrupado — σ={SIGMA_CLUST}, padres={N_PADRES_EXP}"),
        (L_rg_e, "#1565C0", f"Regular — δ={DELTA_REG}")]):
    ax.fill_between(radios, L_env_min, L_env_max, alpha=0.2, color='gray')
    ax.plot(radios, L_env_min, 'gray', linewidth=0.8, linestyle='--')
    ax.plot(radios, L_env_max, 'gray', linewidth=0.8, linestyle='--')
    ax.axhline(0, color='black', linewidth=1, linestyle=':')
    ax.plot(radios, L_o, color=color, linewidth=2.5)
    ax.set_xlabel('Radio r', fontsize=12); ax.set_ylabel('L(r)', fontsize=12)
    ax.set_title(titulo, fontsize=12); ax.grid(True, alpha=0.2); ax.set_xlim(0, 0.25)
plt.suptitle("Experimento — K de Ripley", fontsize=13)
plt.tight_layout(); plt.show()